# FHIR R4 reusable Raw + Bronze ingestion

This parameterized Fabric notebook ingests one resource type per run. The Fabric Data Pipeline calls it sequentially for Patient, Encounter, Observation, and Condition. It preserves response bodies in Lakehouse Files, writes page-level audit metadata, merges unique resource versions into Bronze, and advances each resource's source watermark only after the response pages have been completed and audit rows persisted.

Attach `lh_fhir_assignment` as the default Lakehouse. In Fabric, mark the configuration cell as a notebook parameter cell if it is not recognized automatically.

In [ ]:
# Pipeline parameters: Fabric Data Pipeline can override these values.
resource_type = "Patient"
initial_lookback_days = 3
page_size = 100
overlap_minutes = 5
max_pages = 1000
request_format = "json"

In [ ]:
import hashlib
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable

FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
SUPPORTED_RESOURCES = {"Patient", "Encounter", "Observation", "Condition"}
RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")
AUDIT_TABLE = "fhir_ingestion_audit"
WATERMARK_TABLE = "fhir_ingestion_watermarks"

spark.conf.set("spark.sql.session.timeZone", "UTC")

if resource_type not in SUPPORTED_RESOURCES:
    raise ValueError(f"Unsupported resource_type={resource_type!r}. Choose from {sorted(SUPPORTED_RESOURCES)}")
if request_format.lower() != "json":
    raise ValueError("This required-scope notebook currently supports JSON. XML is an optional extension.")
if page_size < 1 or max_pages < 1 or initial_lookback_days < 1 or overlap_minutes < 0:
    raise ValueError("page_size/max_pages/initial_lookback_days must be positive and overlap_minutes cannot be negative.")

def utc_now():
    return datetime.now(timezone.utc)

def iso_utc(value):
    if value is None or str(value).strip() == "":
        return None
    text = str(value).strip()
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)

def table_exists(name):
    return spark.catalog.tableExists(name)

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("resource_type", StringType(), False),
    StructField("page_number", IntegerType(), False),
    StructField("api_call_timestamp", StringType(), True),
    StructField("extraction_timestamp", StringType(), True),
    StructField("saved_timestamp", StringType(), True),
    StructField("api_url_or_params", StringType(), True),
    StructField("raw_file_path", StringType(), True),
    StructField("metadata_file_path", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("records_received", IntegerType(), True),
    StructField("page_status", StringType(), True),
    StructField("run_status", StringType(), True),
    StructField("error_message", StringType(), True)
])

def persist_audit(rows):
    if not rows:
        return
    source = spark.createDataFrame(rows, schema=audit_schema).dropDuplicates(
        ["run_id", "resource_type", "page_number", "api_call_timestamp"]
    )
    if not table_exists(AUDIT_TABLE):
        source.write.format("delta").mode("overwrite").saveAsTable(AUDIT_TABLE)
    else:
        target = DeltaTable.forName(spark, AUDIT_TABLE)
        condition = "t.run_id = s.run_id AND t.resource_type = s.resource_type AND t.page_number = s.page_number AND t.api_call_timestamp = s.api_call_timestamp"
        (target.alias("t").merge(source.alias("s"), condition)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

run_started = utc_now()
run_id = run_started.strftime("%Y%m%dT%H%M%S%fZ")
extraction_date = run_started.strftime("%Y-%m-%d")
resource_endpoint = f"{FHIR_BASE_URL}/{resource_type}"
previous_watermark = None
if table_exists(WATERMARK_TABLE):
    wm_rows = (spark.table(WATERMARK_TABLE)
               .filter(F.col("resource_type") == resource_type)
               .orderBy(F.col("watermark_updated_at_utc").desc())
               .limit(1).collect())
    if wm_rows:
        previous_watermark = wm_rows[0]["last_successful_source_updated_utc"]

if previous_watermark:
    cutoff_dt = iso_utc(previous_watermark) - timedelta(minutes=int(overlap_minutes))
    cutoff_reason = "persisted watermark minus overlap"
else:
    cutoff_dt = run_started - timedelta(days=int(initial_lookback_days))
    cutoff_reason = f"initial lookback of {initial_lookback_days} days"
cutoff_text = cutoff_dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print("Run ID:", run_id)
print("Resource:", resource_type)
print("Incremental cutoff UTC:", cutoff_text, "(", cutoff_reason, ")")
print("Page size:", page_size, "Max pages:", max_pages)

session = requests.Session()
session.headers.update({"Accept": "application/fhir+json"})
current_url = resource_endpoint
current_params = {"_count": int(page_size), "_format": "json", "_lastUpdated": f"ge{cutoff_text}"}
page_count = 0
resource_records = []
audit_records = []
raw_saved_paths = []
current_attempt_url = resource_endpoint + "?" + urlencode(current_params)
current_call_timestamp = None
current_http_status = None
bronze_table = f"bronze_fhir_{resource_type.lower()}"
bronze_schema = StructType([
    StructField("resource_type", StringType(), False),
    StructField("resource_id", StringType(), False),
    StructField("source_last_updated", StringType(), True),
    StructField("resource_json", StringType(), False),
    StructField("content_hash", StringType(), False),
    StructField("extraction_date", StringType(), True),
    StructField("extraction_timestamp", StringType(), True),
    StructField("api_url_or_params", StringType(), True),
    StructField("raw_file_path", StringType(), False),
    StructField("ingestion_run_id", StringType(), True),
    StructField("saved_timestamp", StringType(), True)
])

try:
    while current_url and page_count < int(max_pages):
        current_call_timestamp = utc_now().isoformat()
        current_attempt_url = current_url
        response = session.get(current_url, params=current_params, timeout=90)
        current_http_status = int(response.status_code)
        actual_url = response.url
        response.raise_for_status()

        bundle = response.json()
        if bundle.get("resourceType") != "Bundle" or bundle.get("type") != "searchset":
            raise ValueError(f"Expected a FHIR searchset Bundle, received resourceType={bundle.get('resourceType')!r}, type={bundle.get('type')!r}")

        page_count += 1
        page_entries = bundle.get("entry", []) or []
        raw_dir = RAW_ROOT / f"resource={resource_type}" / f"extraction_date={extraction_date}" / f"run_id={run_id}"
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = raw_dir / f"page_{page_count:05d}.json"
        metadata_path = raw_dir / f"page_{page_count:05d}.metadata.json"

        # Preserve the original successful response body; write enrichment to a separate sidecar.
        raw_path.write_bytes(response.content)
        saved_timestamp = utc_now().isoformat()
        raw_saved_paths.append(str(raw_path))
        page_metadata = {
            "run_id": run_id, "resource_type": resource_type, "page_number": page_count,
            "api_call_timestamp": current_call_timestamp,
            "extraction_timestamp": current_call_timestamp,
            "saved_timestamp": saved_timestamp, "api_url_or_params": actual_url,
            "raw_file_path": str(raw_path), "http_status": current_http_status,
            "records_received": len(page_entries), "incremental_cutoff_utc": cutoff_text
        }
        metadata_path.write_text(json.dumps(page_metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        audit_records.append({
            "run_id": run_id, "resource_type": resource_type, "page_number": page_count,
            "api_call_timestamp": current_call_timestamp, "extraction_timestamp": current_call_timestamp,
            "saved_timestamp": saved_timestamp, "api_url_or_params": actual_url,
            "raw_file_path": str(raw_path), "metadata_file_path": str(metadata_path),
            "http_status": current_http_status, "records_received": len(page_entries),
            "page_status": "RAW_SAVED", "run_status": "IN_PROGRESS", "error_message": None
        })

        for entry in page_entries:
            resource = entry.get("resource") or {}
            if resource.get("resourceType") != resource_type:
                continue
            resource_id = resource.get("id")
            if not resource_id:
                print("WARNING: skipping resource without an id; the original Bundle is retained in Raw.")
                continue
            canonical_json = json.dumps(resource, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
            content_hash = hashlib.sha256(canonical_json.encode("utf-8")).hexdigest()
            resource_records.append({
                "resource_type": resource_type, "resource_id": str(resource_id),
                "source_last_updated": resource.get("meta", {}).get("lastUpdated"),
                "resource_json": canonical_json, "content_hash": content_hash,
                "extraction_date": extraction_date, "extraction_timestamp": current_call_timestamp,
                "api_url_or_params": actual_url, "raw_file_path": str(raw_path),
                "ingestion_run_id": run_id, "saved_timestamp": saved_timestamp
            })

        next_url = next((link.get("url") for link in bundle.get("link", []) if link.get("relation") == "next"), None)
        current_url = next_url
        current_params = None  # The server-supplied next URL includes paging parameters.
        current_attempt_url = current_url or actual_url

    if current_url:
        raise RuntimeError(f"Pagination safety limit reached ({max_pages} pages). Watermark was not advanced; increase max_pages and rerun.")

    resource_df = spark.createDataFrame(resource_records, schema=bronze_schema) if resource_records else spark.createDataFrame([], schema=bronze_schema)
    if not table_exists(bronze_table):
        resource_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
    elif resource_records:
        source = resource_df.dropDuplicates(["resource_type", "resource_id", "content_hash"])
        target = DeltaTable.forName(spark, bronze_table)
        condition = "t.resource_type = s.resource_type AND t.resource_id = s.resource_id AND t.content_hash = s.content_hash"
        (target.alias("t").merge(source.alias("s"), condition)
         .whenNotMatchedInsertAll().execute())

    for row in audit_records:
        row["run_status"] = "SUCCESS"
    persist_audit(audit_records)

    valid_source_times = [iso_utc(r["source_last_updated"]) for r in resource_records if r.get("source_last_updated")]
    valid_source_times = [x for x in valid_source_times if x is not None]
    if valid_source_times:
        max_source_updated = max(valid_source_times).isoformat(timespec="microseconds")
        watermark_row = [{
            "resource_type": resource_type,
            "last_successful_source_updated_utc": max_source_updated,
            "last_successful_run_id": run_id,
            "watermark_updated_at_utc": utc_now().isoformat()
        }]
        watermark_schema = StructType([
            StructField("resource_type", StringType(), False),
            StructField("last_successful_source_updated_utc", StringType(), False),
            StructField("last_successful_run_id", StringType(), False),
            StructField("watermark_updated_at_utc", StringType(), False)
        ])
        wm_df = spark.createDataFrame(watermark_row, schema=watermark_schema)
        if not table_exists(WATERMARK_TABLE):
            wm_df.write.format("delta").mode("overwrite").saveAsTable(WATERMARK_TABLE)
        else:
            wm_target = DeltaTable.forName(spark, WATERMARK_TABLE)
            (wm_target.alias("t").merge(wm_df.alias("s"), "t.resource_type = s.resource_type")
             .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    else:
        print("No source lastUpdated timestamp found; watermark was not advanced.")

    print("\nIngestion completed successfully.")
    print("Pages saved:", page_count)
    print("Resource records observed:", len(resource_records))
    print("Raw pages:", len(raw_saved_paths))
    print("Bronze table:", bronze_table)
    print("Audit table:", AUDIT_TABLE)
    print("Watermark table:", WATERMARK_TABLE)

except Exception as exc:
    failed_page_number = page_count + 1
    audit_records.append({
        "run_id": run_id, "resource_type": resource_type, "page_number": failed_page_number,
        "api_call_timestamp": current_call_timestamp or utc_now().isoformat(),
        "extraction_timestamp": current_call_timestamp or utc_now().isoformat(),
        "saved_timestamp": None, "api_url_or_params": current_attempt_url,
        "raw_file_path": None, "metadata_file_path": None, "http_status": current_http_status or 0,
        "records_received": 0, "page_status": "RUN_FAILURE", "run_status": "FAILED",
        "error_message": str(exc)[:4000]
    })
    for row in audit_records:
        row["run_status"] = "FAILED"
    try:
        persist_audit(audit_records)
    except Exception as audit_exc:
        print("WARNING: Failed to persist audit rows:", str(audit_exc))
    print("Ingestion failed. Review the persisted audit rows and Raw sidecars before retrying.")
    raise
finally:
    session.close()